# Deepfake Detection Using Deep Learning
**Emirhan Er — Senior Project**

EfficientNet-B4 fine-tuned for binary (real / fake) classification.  
**No uploads needed** — everything runs inside Colab.

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── 0. Connect VS Code to this Colab runtime ───────────────────────────────
# Run this cell FIRST, then paste the printed URL into VS Code
# Get your free token at: https://dashboard.ngrok.com/get-started/your-authtoken

!pip install pyngrok -q

import subprocess, time, urllib
from pyngrok import ngrok

NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- paste your token here
ngrok.set_auth_token(NGROK_TOKEN)

# Start Jupyter server
subprocess.Popen([
    'jupyter', 'notebook',
    '--no-browser',
    '--port=8888',
    '--NotebookApp.allow_origin=*',
    '--NotebookApp.ip=0.0.0.0',
    '--NotebookApp.disable_check_xsrf=True'
])
time.sleep(4)

# Create public tunnel
tunnel = ngrok.connect(8888)
print('\n' + '='*60)
print('COPY THIS URL INTO VS CODE:')

# Get the token
result = subprocess.run(['jupyter', 'notebook', 'list'], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'token=' in line:
        token = line.split('token=')[1].split(' ')[0]
        print(f"{tunnel.public_url}/?token={token}")
print('='*60)
print('\nIn VS Code: Ctrl+Shift+P → "Jupyter: Specify Jupyter Server" → Existing → paste URL above')

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
!pip install -q timm albumentations grad-cam seaborn

In [ ]:
# ── 2. Kaggle credentials & dataset download ───────────────────────────────
import os, json

# Paste your Kaggle credentials here
KAGGLE_USERNAME = "emirhaner68"
KAGGLE_KEY      = "YOUR_NEW_KAGGLE_KEY_HERE"   # <-- replace after regenerating

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/
!unzip -q /content/140k-real-and-fake-faces.zip -d /content/data/
print('Dataset ready.')

In [ ]:
# ── 3. Build CSV manifests (images already face-cropped, no MTCNN needed) ─
import glob, pandas as pd

RAW  = '/content/data/real_vs_fake/real-vs-fake'
MDIR = '/content/manifests'
os.makedirs(MDIR, exist_ok=True)

for split, out in [('train','train'), ('valid','val'), ('test','test')]:
    records = []
    for cls, label in [('real', 0), ('fake', 1)]:
        for p in glob.glob(f'{RAW}/{split}/{cls}/*.jpg'):
            records.append({'path': p, 'label': label})
    df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
    df.to_csv(f'{MDIR}/{out}.csv', index=False)
    print(f'{out:6s}: {len(df):>7,}  real={sum(df.label==0):,}  fake={sum(df.label==1):,}')

In [ ]:
# ── 4. Dataset preview ─────────────────────────────────────────────────────
import cv2, numpy as np, matplotlib.pyplot as plt

train_df = pd.read_csv(f'{MDIR}/train.csv')
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes.flat,
    pd.concat([train_df[train_df.label==0].head(4),
               train_df[train_df.label==1].head(4)]).iterrows()):
    img = cv2.cvtColor(cv2.imread(row['path']), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(['Real','Fake'][row['label']]); ax.axis('off')
plt.suptitle('Sample images from dataset', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ── 5. Dataset class ───────────────────────────────────────────────────────
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)
IMG_SIZE = 380

def build_transforms(split):
    if split == 'train':
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.4),
            A.ImageCompression(quality_range=(70,100), p=0.3),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

class DeepfakeDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.cvtColor(cv2.imread(row['path']), cv2.COLOR_BGR2RGB)
        return self.transform(image=img)['image'], int(row['label'])

BS = 32
train_loader = DataLoader(DeepfakeDataset(f'{MDIR}/train.csv', build_transforms('train')),
                          batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(DeepfakeDataset(f'{MDIR}/val.csv',   build_transforms('val')),
                          batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(DeepfakeDataset(f'{MDIR}/test.csv',  build_transforms('test')),
                          batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}')

In [ ]:
# ── 6. Model (EfficientNet-B4, ImageNet pretrained) ────────────────────────
import timm, torch.nn as nn

class DeepfakeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone   = timm.create_model('tf_efficientnet_b4.ns_jft_in1k',
                                            pretrained=True, num_classes=0, global_pool='avg')
        self.classifier = nn.Sequential(nn.Dropout(0.3),
                                        nn.Linear(self.backbone.num_features, 2))
    def forward(self, x): return self.classifier(self.backbone(x))
    def feature_layer(self):  return self.backbone.conv_head

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = DeepfakeClassifier().to(device)
params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Device: {device}  |  Parameters: {params:.1f}M')

In [ ]:
# ── 7. Training loop ───────────────────────────────────────────────────────
import math
from torch.amp import GradScaler, autocast
from tqdm.notebook import tqdm

EPOCHS   = 15
LR       = 1e-4
PATIENCE = 4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scaler    = GradScaler('cuda')

total_steps  = EPOCHS * len(train_loader)
warmup_steps = 2 * len(train_loader)
def lr_lambda(step):
    if step < warmup_steps: return step / max(1, warmup_steps)
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.0, 0.5 * (1 + math.cos(math.pi * p)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

os.makedirs('/content/checkpoints', exist_ok=True)
history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
best_val_loss, no_improve = float('inf'), 0

for epoch in range(1, EPOCHS+1):
    # ── train
    model.train()
    tloss = tcorr = ttotal = 0
    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [train]', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast('cuda'):
            loss = criterion(model(imgs), labels) / 1
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        with torch.no_grad():
            preds = model(imgs).argmax(1)
        tloss  += loss.item() * imgs.size(0)
        tcorr  += (preds == labels).sum().item()
        ttotal += imgs.size(0)

    # ── val
    model.eval(); vloss = vcorr = vtotal = 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} [val]', leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            with autocast('cuda'):
                logits = model(imgs)
                vloss += criterion(logits, labels).item() * imgs.size(0)
            vcorr  += (logits.argmax(1) == labels).sum().item()
            vtotal += imgs.size(0)

    ta, va = tcorr/ttotal, vcorr/vtotal
    tl, vl = tloss/ttotal, vloss/vtotal
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    print(f'Epoch {epoch:2d}/{EPOCHS}  train loss={tl:.4f} acc={ta:.4f}  val loss={vl:.4f} acc={va:.4f}')

    if vl < best_val_loss:
        best_val_loss = vl; no_improve = 0
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),'val_acc':va},
                   '/content/checkpoints/best_model.pt')
        print(f'  ✓ Checkpoint saved')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}'); break

print('Training complete.')

In [ ]:
# ── 8. Training curves ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, len(history['train_loss'])+1)
axes[0].plot(ep, history['train_loss'], label='Train'); axes[0].plot(ep, history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(ep, history['train_acc'],  label='Train'); axes[1].plot(ep, history['val_acc'],  label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.savefig('/content/training_curves.png', dpi=150); plt.show()

In [ ]:
# ── 9. Evaluation on test set ──────────────────────────────────────────────
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)
import seaborn as sns

# Load best checkpoint
ckpt = torch.load('/content/checkpoints/best_model.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_labels, all_preds, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Evaluating'):
        imgs = imgs.to(device)
        with autocast('cuda'):
            logits = model(imgs)
        probs = torch.softmax(logits, 1)[:,1].cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

labels_np = np.array(all_labels)
preds_np  = np.array(all_preds)
probs_np  = np.array(all_probs)

metrics = {
    'accuracy':  accuracy_score(labels_np, preds_np),
    'precision': precision_score(labels_np, preds_np),
    'recall':    recall_score(labels_np, preds_np),
    'f1':        f1_score(labels_np, preds_np),
    'auc':       roc_auc_score(labels_np, probs_np),
}
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.4f}')
print()
print(classification_report(labels_np, preds_np, target_names=['Real','Fake']))

In [ ]:
# ── 10. Confusion matrix & ROC curve ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(labels_np, preds_np)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real','Fake'], yticklabels=['Real','Fake'], ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(labels_np, probs_np)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f"AUC = {metrics['auc']:.4f}")
axes[1].plot([0,1],[0,1], 'grey', linestyle='--', lw=1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('/content/evaluation_plots.png', dpi=150)
plt.show()

In [ ]:
# ── 11. Grad-CAM visualisation ─────────────────────────────────────────────
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from IPython.display import display
from PIL import Image as PILImage

CLASS_NAMES = {0: 'Real', 1: 'Fake'}

def gradcam_figure(img_path, model, device):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_res = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_f32 = img_res.astype(np.float32) / 255.0

    tensor = build_transforms('test')(image=img_res)['image'].unsqueeze(0).to(device)

    with torch.no_grad():
        probs = torch.softmax(model(tensor), 1)[0]
    pred  = probs.argmax().item()
    conf  = probs[pred].item()

    with GradCAM(model=model, target_layers=[model.feature_layer()]) as cam:
        mask = cam(tensor, targets=[ClassifierOutputTarget(pred)])[0]
    vis = show_cam_on_image(img_f32, mask, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img_res);  axes[0].set_title('Input');              axes[0].axis('off')
    axes[1].imshow(vis);      axes[1].set_title(f'Grad-CAM → {CLASS_NAMES[pred]} ({conf:.1%})'); axes[1].axis('off')
    plt.suptitle(f'Prediction: {CLASS_NAMES[pred]}  ({conf:.1%} confidence)', fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close()

# Sample 4 real + 4 fake from test set
test_df = pd.read_csv(f'{MDIR}/test.csv')
samples = (
    test_df[test_df.label==0].sample(4, random_state=42)['path'].tolist() +
    test_df[test_df.label==1].sample(4, random_state=42)['path'].tolist()
)
for path in samples:
    gradcam_figure(path, model, device)

In [ ]:
# ── 12. Download results to your computer ──────────────────────────────────
import shutil, json
from google.colab import files

# Save metrics
with open('/content/metrics.json', 'w') as f:
    json.dump({k: round(float(v), 6) for k, v in metrics.items()}, f, indent=2)

# Bundle everything into a zip
os.makedirs('/content/results', exist_ok=True)
shutil.copy('/content/checkpoints/best_model.pt', '/content/results/best_model.pt')
shutil.copy('/content/training_curves.png',        '/content/results/training_curves.png')
shutil.copy('/content/evaluation_plots.png',       '/content/results/evaluation_plots.png')
shutil.copy('/content/metrics.json',               '/content/results/metrics.json')
shutil.make_archive('/content/deepfake_results', 'zip', '/content/results')

files.download('/content/deepfake_results.zip')
print('Downloaded.')